In [1]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

df = pd.read_csv("ner.csv")

import ast

sentences = df['Sentence']
tags = df['Tag']

# Fill NaN values in the 'tags' column with an empty list string before applying literal_eval
parsed_tags = tags.fillna('[]').apply(ast.literal_eval)

x_tokenizer = Tokenizer()
x_tokenizer.fit_on_texts(sentences)
x_data = x_tokenizer.texts_to_sequences(sentences)
X = pad_sequences(x_data, padding='post')
max_x_seq = X.shape[1]
# Determine the maximum sequence length from X to ensure Y matches
y_tokenizer = Tokenizer()
y_tokenizer.fit_on_texts(parsed_tags) # Use parsed_tags here
y_data = y_tokenizer.texts_to_sequences(parsed_tags) # Use parsed_tags here
Y = pad_sequences(y_data, padding='post',maxlen=max_x_seq) # Pad Y to match X's sequence length
x_train , x_test , y_train, y_test = train_test_split(X,Y, test_size= 0.2)


In [5]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense

input_layer = Input(shape=(max_x_seq,))
embedding = Embedding(
    input_dim=(len(x_tokenizer.word_index)+1),
    output_dim=128,
)(input_layer)

bilstm = Bidirectional(
    LSTM(64, return_sequences=True)
)(embedding)

output = Dense((len(y_tokenizer.word_index)+1), activation="softmax")(bilstm)

model = Model(input_layer, output)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

model.fit(
    x_train,
   y_train,
   validation_data = (x_test,y_test),
    epochs=1
)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 89)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, 89, 128)        │     2,964,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 89, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 89, 18)         │         2,322 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,065,234 (11.69 MB)

 Trainable params: 3,065,234 (11.69 MB)

 Non-trainable params: 0 (0.00 B)

766/766 ━━━━━━━━━━━━━━━━━━━━ 131s 165ms/step - accuracy: 0.9262 - loss: 0.3064 - val_accuracy: 0.9624 - val_loss: 0.1219


In [34]:
import numpy as np

def predict_tags(text):
    # Tokenize the input text
    tokenized_text = x_tokenizer.texts_to_sequences([text])
    # Pad the tokenized text to the same length as training data
    padded_text = pad_sequences(tokenized_text, padding='post', maxlen=max_x_seq)

    # Make prediction
    prediction = model.predict(padded_text)
    # Get the index of the tag with the highest probability for each word
    predicted_tag_indices = np.argmax(prediction, axis=-1)[0]
    for i in range(len(padded_text[0])):
        idx2 = predicted_tag_indices[i]
        idx1 = padded_text[0][i]
        if idx2 != 0:
          print(f"{x_tokenizer.index_word[idx1]}: {y_tokenizer.index_word[idx2]}")
    return padded_text , predicted_tag_indices
predict_tags('india pakistan afghanistan')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
india: b-geo
pakistan: b-geo
afghanistan: b-geo


(array([[247, 132, 109,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],
       dtype=int32),
 array([2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0]))